# 02 Analysis: Where Is Toronto Construction Demand Growing?

**Business task:** identify where in Toronto construction activity is growing, what kind of work it is, and how quickly permits move, so a building supply company's sales team can focus territory and product mix on the strongest demand.

## Findings summary

_To be written from the outputs of the first full run. Every number here will come from a cell below._

## How demand is measured

Decisions made before any numbers were seen, so they cannot be tuned to give a nicer answer:

| Decision | Choice | Why |
|---|---|---|
| Cancelled permits | Excluded | A cancelled permit is unlikely to lead to material purchases |
| Trade permits tied to a building permit (`is_related_permit`) | Excluded from counts | Otherwise one project counts several times (plumbing, HVAC, drains). They are already covered by the main permit |
| Declared value | Only rows with `cost_declared` | 0 and placeholder costs mean "not declared", not "free" |
| Years compared | Full calendar years only | The current year is partial and would look like a drop |
| Growth | Last two full years vs the two years before | Two-year windows smooth out one-off big projects |
| Small areas | FSAs need at least 50 permits in the base window to get a growth rate | Growth from 5 to 10 permits is "+100%" but means little |
| Approval time | Issued, non-revised permits; median, not mean | Approval days are heavily right-skewed; a few multi-year permits would drag the mean |

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CLEAN = ROOT / "data" / "clean"
TABLES = ROOT / "outputs" / "summary_tables"
FIGS = ROOT / "outputs" / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGS.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.1f}".format)

# Chart style: one consistent, colorblind-checked palette, used in a fixed
# order so a category keeps its colour across every chart.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e",
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "grid.color": "#e6e5e0", "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
    "lines.linewidth": 2, "font.size": 10, "axes.titlesize": 12,
    "axes.titleweight": "bold", "axes.titlelocation": "left",
})

def save_table(df, name):
    # Every summary table is saved for Tableau and for the README.
    df.to_csv(TABLES / f"{name}.csv", index=True)
    print(f"saved outputs/summary_tables/{name}.csv ({len(df):,} rows)")

## 1. Load the clean data and define the analysis set

In [ ]:
date_cols = ["APPLICATION_DATE", "ISSUED_DATE", "COMPLETED_DATE", "app_year_month"]
permits = pd.read_csv(
    CLEAN / "permits_clean.csv",
    parse_dates=date_cols,
    dtype={"PERMIT_NUM": str, "REVISION_NUM": str, "project_id": str, "fsa": str, "POSTAL": str},
    low_memory=False,
)
# Make sure the flag columns are real booleans whatever pandas inferred.
for col in ["cost_declared", "is_cancelled", "is_revision", "is_related_permit"]:
    permits[col] = permits[col].astype(str).str.lower().eq("true")

print(f"Clean permits loaded: {len(permits):,}")

# The data runs to the latest daily refresh, so the last month is partial.
# Work out the last FULL month and the last FULL year from the data itself.
last_date = permits["APPLICATION_DATE"].max()
LAST_FULL_MONTH = (last_date.to_period("M") - 1).to_timestamp()
LAST_FULL_YEAR = LAST_FULL_MONTH.year if LAST_FULL_MONTH.month == 12 else LAST_FULL_MONTH.year - 1
FULL_YEARS = list(range(2017, LAST_FULL_YEAR + 1))
RECENT = [LAST_FULL_YEAR - 1, LAST_FULL_YEAR]          # e.g. 2024-2025
BASE = [LAST_FULL_YEAR - 3, LAST_FULL_YEAR - 2]        # e.g. 2022-2023

print("Latest application date:", last_date.date())
print("Last full month:        ", LAST_FULL_MONTH.strftime("%Y-%m"))
print("Full years:             ", FULL_YEARS[0], "to", FULL_YEARS[-1])
print("Growth windows:          base", BASE, "vs recent", RECENT)

In [ ]:
# The analysis set: permits that represent real, distinct demand.
demand = permits[~permits["is_cancelled"] & ~permits["is_related_permit"]].copy()

print(f"All clean permits:                {len(permits):,}")
print(f"  minus cancelled:                {permits['is_cancelled'].sum():,}")
print(f"  minus trade permits (related):  {(permits['is_related_permit'] & ~permits['is_cancelled']).sum():,}")
print(f"Demand permits:                   {len(demand):,}")
print(f"  with a declared cost:           {demand['cost_declared'].sum():,}")
print(f"  with an FSA:                    {demand['fsa'].notna().sum():,}")

# Value column that only holds declared costs, so sums and medians ignore the rest.
demand["declared_value"] = demand["est_cost"].where(demand["cost_declared"])

## Q1. Where is permit activity concentrated and growing?

For each FSA: permits and declared value in the base and recent windows, growth in permit count, and the FSA's share of the city's recent value.

Growth is measured on **permit counts**, not value. A single tower can add hundreds of millions of dollars to one FSA in one year, which would make value growth mostly a story about one project. Value is shown alongside so both views are visible.

In [ ]:
fsa_df = demand[demand["fsa"].notna() & demand["app_year"].isin(BASE + RECENT)].copy()
fsa_df["window"] = np.where(fsa_df["app_year"].isin(RECENT), "recent", "base")

fsa = fsa_df.pivot_table(index="fsa", columns="window", values="PERMIT_NUM",
                         aggfunc="count", fill_value=0)
fsa.columns = [f"permits_{c}" for c in fsa.columns]

value = fsa_df.pivot_table(index="fsa", columns="window", values="declared_value",
                           aggfunc="sum", fill_value=0)
value.columns = [f"value_{c}" for c in value.columns]

new_build = (fsa_df[(fsa_df["window"] == "recent") & (fsa_df["work_category"] == "New construction")]
             .groupby("fsa").size().rename("new_construction_recent"))

fsa = fsa.join(value).join(new_build).fillna({"new_construction_recent": 0})
fsa["growth_pct"] = np.where(
    fsa["permits_base"] >= 50,
    (fsa["permits_recent"] / fsa["permits_base"] - 1) * 100,
    np.nan,  # too few permits in the base window for a meaningful rate
)
fsa["share_of_city_value_recent_pct"] = fsa["value_recent"] / fsa["value_recent"].sum() * 100
fsa = fsa.sort_values("value_recent", ascending=False)

print(f"FSAs: {len(fsa)}, with a growth rate: {fsa['growth_pct'].notna().sum()}")
city_growth = (fsa["permits_recent"].sum() / fsa["permits_base"].sum() - 1) * 100
print(f"City-wide permit growth, {BASE} -> {RECENT}: {city_growth:+.1f}%")
save_table(fsa, "q1_fsa_growth")

In [ ]:
# Concentration: how much of the recent declared value sits in the top FSAs?
top10_share = fsa["share_of_city_value_recent_pct"].head(10).sum()
print(f"Top 10 FSAs hold {top10_share:.1f}% of recent declared value "
      f"({len(fsa)} FSAs in total)")
print()
print("Top 10 FSAs by recent declared value:")
display(fsa.head(10)[["permits_recent", "value_recent", "share_of_city_value_recent_pct",
                      "new_construction_recent", "growth_pct"]])

In [ ]:
print("Top 15 FSAs by permit growth (min 50 permits in base window):")
top_growth = fsa.dropna(subset=["growth_pct"]).sort_values("growth_pct", ascending=False)
display(top_growth.head(15)[["permits_base", "permits_recent", "growth_pct",
                             "value_recent", "new_construction_recent"]])
print("Bottom 5:")
display(top_growth.tail(5)[["permits_base", "permits_recent", "growth_pct"]])

In [ ]:
# CHECK: is an FSA's recent value driven by one project? If the largest single
# permit is most of an FSA's value, treat that FSA's value ranking with care.
recent = fsa_df[fsa_df["window"] == "recent"]
biggest = recent.groupby("fsa")["declared_value"].max()
fsa["largest_permit_share_pct"] = biggest / fsa["value_recent"] * 100
display(fsa.head(10)[["value_recent", "largest_permit_share_pct"]])

In [ ]:
plot = top_growth.head(15).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.barh(plot.index, plot["growth_pct"], color=PALETTE[0], height=0.6)
ax.axvline(city_growth, color="#52514e", linewidth=1, linestyle="--")
ax.text(city_growth, len(plot) - 0.3, f" city {city_growth:+.0f}%", color="#52514e", va="bottom")
ax.set_xlabel(f"Change in permits, {BASE[0]}-{BASE[1]} to {RECENT[0]}-{RECENT[1]} (%)")
ax.set_title("FSAs with the fastest permit growth")
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(FIGS / "q1_fsa_growth.png", dpi=150)
plt.show()

## Q2. What kind of work is it?

Split by sector (residential, non-residential, mixed use) and work category, by permit count and by declared value, over the full period and year by year.

In [ ]:
full = demand[demand["app_year"].isin(FULL_YEARS)]

mix_count = pd.crosstab(full["sector"], full["work_category"], margins=True, margins_name="Total")
mix_value = full.pivot_table(index="sector", columns="work_category", values="declared_value",
                             aggfunc="sum", margins=True, margins_name="Total") / 1e6

print("Permits by sector and work category:")
display(mix_count)
print("Declared value ($ millions):")
display(mix_value)

share_count = (mix_count / mix_count.loc["Total", "Total"] * 100).round(1)
share_value = (mix_value / mix_value.loc["Total", "Total"] * 100).round(1)
print("Share of permits (%):")
display(share_count)
print("Share of declared value (%):")
display(share_value)

save_table(mix_count, "q2_mix_permits")
save_table(mix_value, "q2_mix_value_millions")

In [ ]:
# How the work mix changes year to year (share of each year's permits).
mix_year = pd.crosstab(full["app_year"], full["work_category"], normalize="index") * 100
display(mix_year.round(1))
save_table(mix_year, "q2_work_mix_by_year_pct")

# Residential share of permits and of value, by year.
res = full.groupby("app_year").apply(lambda g: pd.Series({
    "residential_permit_share_pct": (g["sector"] == "Residential").mean() * 100,
    "residential_value_share_pct": g.loc[g["sector"] == "Residential", "declared_value"].sum()
                                   / g["declared_value"].sum() * 100,
}), include_groups=False)
display(res.round(1))
save_table(res, "q2_residential_share_by_year")

## Q3. How long do permits take from application to issuance?

Only issued, non-revised permits (revision rows can carry a later application date). We report the median and the middle 50% range (25th to 75th percentile), because a few permits take years.

**Important bias:** for recent application years, permits that are still waiting are not in the data yet as issued. Only the fast ones have finished, so recent years will look faster than they really are. The year table shows the share still not issued so this is visible.

In [ ]:
issued = demand[demand["approval_days"].notna() & ~demand["is_revision"]]
print(f"Issued, non-revised demand permits: {len(issued):,}")

def approval_summary(df, by, min_n=100):
    out = df.groupby(by)["approval_days"].agg(
        permits="count",
        median="median",
        p25=lambda s: s.quantile(0.25),
        p75=lambda s: s.quantile(0.75),
        p90=lambda s: s.quantile(0.90),
    )
    return out[out["permits"] >= min_n].sort_values("median")

by_cat = approval_summary(issued, "work_category")
by_sector = approval_summary(issued, "sector")
by_type = approval_summary(issued, "PERMIT_TYPE")
print("By work category:"); display(by_cat)
print("By sector:"); display(by_sector)
print("By permit type (min 100 permits):"); display(by_type)
save_table(by_cat, "q3_approval_by_work_category")
save_table(by_sector, "q3_approval_by_sector")
save_table(by_type, "q3_approval_by_permit_type")

In [ ]:
# By application year, with the share of that year's permits not yet issued.
by_year = approval_summary(issued, "app_year", min_n=1).sort_index()
not_issued = (demand[~demand["is_revision"]].groupby("app_year")["ISSUED_DATE"]
              .apply(lambda s: s.isna().mean() * 100).rename("not_yet_issued_pct"))
by_year = by_year.join(not_issued)
display(by_year)
save_table(by_year, "q3_approval_by_year")

In [ ]:
# By FSA. Work type drives approval time, so compare FSAs within the two
# largest categories rather than mixing a new tower with a deck permit.
fsa_rows = []
for cat in ["Renovation / alteration", "New construction"]:
    t = approval_summary(issued[issued["work_category"] == cat], "fsa", min_n=30)
    t["work_category"] = cat
    fsa_rows.append(t)
by_fsa = pd.concat(fsa_rows)
save_table(by_fsa, "q3_approval_by_fsa")

for cat in ["Renovation / alteration", "New construction"]:
    t = by_fsa[by_fsa["work_category"] == cat]
    print(f"{cat}: {len(t)} FSAs with 30+ permits, "
          f"median ranges from {t['median'].min():.0f} to {t['median'].max():.0f} days")
    display(pd.concat([t.head(5), t.tail(5)])[["permits", "median", "p25", "p75"]])

In [ ]:
plot = by_cat.sort_values("median")
fig, ax = plt.subplots(figsize=(7, 4))
y = np.arange(len(plot))
ax.hlines(y, plot["p25"], plot["p75"], color="#86b6ef", linewidth=6, label="middle 50% of permits")
ax.plot(plot["median"], y, "o", color=PALETTE[0], markersize=8, label="median")
for yi, m in zip(y, plot["median"]):
    ax.text(m, yi - 0.22, f"{m:.0f}", ha="center", va="top", color="#0b0b0b", fontsize=9)
ax.set_yticks(y, plot.index)
ax.set_xlabel("Days from application to issuance")
ax.set_title("Approval time by work category")
ax.set_ylim(-0.7, len(plot) - 0.4)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=False)
ax.grid(axis="y", visible=False)
fig.tight_layout()
fig.savefig(FIGS / "q3_approval_by_category.png", dpi=150)
plt.show()

## Q4. How have volumes and values changed, and is there a seasonal pattern?

In [ ]:
annual = full.groupby("app_year").agg(
    permits=("PERMIT_NUM", "count"),
    declared_value_millions=("declared_value", lambda s: s.sum() / 1e6),
    median_declared_cost=("declared_value", "median"),
    new_construction=("work_category", lambda s: (s == "New construction").sum()),
)
annual["permits_yoy_pct"] = annual["permits"].pct_change() * 100
annual["value_yoy_pct"] = annual["declared_value_millions"].pct_change() * 100
display(annual)
save_table(annual, "q4_annual_totals")

In [ ]:
# Monthly permits by work category (full months only), for trend charts.
monthly = (demand[demand["app_year_month"] <= LAST_FULL_MONTH]
           .pivot_table(index="app_year_month", columns="work_category",
                        values="PERMIT_NUM", aggfunc="count", fill_value=0))
save_table(monthly, "q4_monthly_by_work_category")

# Seasonality: each month's share of its year's permits, averaged over full
# years. Index 100 = an average month; 120 = 20% busier than average.
m = full.groupby(["app_year", "app_month"]).size().rename("permits").reset_index()
m["index"] = m["permits"] / m.groupby("app_year")["permits"].transform("mean") * 100
season = m.groupby("app_month")["index"].agg(["mean", "min", "max"]).round(1)
season.index = pd.to_datetime(season.index, format="%m").strftime("%b")
display(season)
save_table(season, "q4_seasonality_index")

In [ ]:
# A second view of seasonality: classical decomposition of the monthly total
# into trend + repeating 12-month pattern + leftover noise. If the seasonal
# swing is small relative to the trend, seasonality matters less for planning.
total_monthly = monthly.sum(axis=1)
total_monthly = total_monthly[total_monthly.index.year.isin(FULL_YEARS)]
decomp = seasonal_decompose(total_monthly, model="additive", period=12)

seasonal_range = decomp.seasonal.max() - decomp.seasonal.min()
print(f"Average monthly permits: {total_monthly.mean():,.0f}")
print(f"Seasonal swing (peak month minus low month): {seasonal_range:,.0f} permits")

fig = decomp.plot()
fig.set_size_inches(8, 6)
fig.tight_layout()
fig.savefig(FIGS / "q4_seasonal_decomposition.png", dpi=150)
plt.show()

In [ ]:
# Annual permits by work category. Building services and Other are left out
# to keep the chart readable; they are in the saved table.
cats = ["Renovation / alteration", "New construction", "Addition", "Demolition"]
yearly = pd.crosstab(full["app_year"], full["work_category"])[cats]

fig, ax = plt.subplots(figsize=(8, 4.5))
for i, cat in enumerate(cats):
    ax.plot(yearly.index, yearly[cat], color=PALETTE[i], marker="o", markersize=5, label=cat)
    ax.text(yearly.index[-1] + 0.15, yearly[cat].iloc[-1], cat, color="#52514e", va="center", fontsize=9)
ax.set_xlim(yearly.index[0] - 0.3, yearly.index[-1] + 2.2)
ax.set_xticks(yearly.index)
ax.set_ylim(0)
ax.set_ylabel("Permits applied for")
ax.set_title("Permits per year by type of work")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.45, -0.1), ncol=4, fontsize=9)
fig.tight_layout()
fig.savefig(FIGS / "q4_annual_by_work_category.png", dpi=150)
plt.show()

## Table for Tableau

One row per FSA, month, sector and work category, with permit count, declared value and median approval days. This is the main data source for the Share phase dashboard (map, monthly trend, approval times, growth areas).

In [ ]:
tableau = (demand[demand["fsa"].notna() & (demand["app_year_month"] <= LAST_FULL_MONTH)]
           .groupby(["fsa", "app_year_month", "sector", "work_category"])
           .agg(permits=("PERMIT_NUM", "count"),
                declared_value=("declared_value", "sum"),
                permits_with_value=("cost_declared", "sum"),
                median_approval_days=("approval_days", "median"))
           .reset_index())
tableau.to_csv(TABLES / "tableau_fsa_month.csv", index=False)
print(f"saved outputs/summary_tables/tableau_fsa_month.csv ({len(tableau):,} rows)")

## Key numbers for the findings summary

In [ ]:
fastest = top_growth.head(3)
print(f"Analysis set: {len(demand):,} demand permits, {FULL_YEARS[0]}-{FULL_YEARS[-1]} full years")
print(f"City-wide permit change {BASE} -> {RECENT}: {city_growth:+.1f}%")
print(f"Top 10 FSAs share of recent declared value: {top10_share:.1f}%")
print("Fastest-growing FSAs:", ", ".join(f"{i} ({g:+.0f}%)" for i, g in fastest["growth_pct"].items()))
print(f"Residential share of permits, full period: {share_count.loc['Residential', 'Total']:.1f}%")
print(f"Residential share of value, full period:   {share_value.loc['Residential', 'Total']:.1f}%")
print("Median approval days by category:", ", ".join(f"{c} {d:.0f}" for c, d in by_cat["median"].items()))
busiest = season["mean"].idxmax(); quietest = season["mean"].idxmin()
print(f"Busiest month: {busiest} (index {season['mean'].max():.0f}); quietest: {quietest} (index {season['mean'].min():.0f})")